In [1]:
import pandas as pd
import numpy as np
# import myfunction as mf
# import myuncertain as mu
# import os
from path_config import *



### 计算thq

In [2]:
import os
from scipy.stats import truncnorm, triang
import xarray as xr
from typing import Callable, List


# =============================
# 工具函数
# =============================
def truncated_normal(mean: np.ndarray, cv: float, lower: np.ndarray, upper: np.ndarray):
    """生成截断正态分布对象"""
    std = mean * cv
    a = (lower - mean) / std
    b = (upper - mean) / std
    return truncnorm(a, b, loc=mean, scale=std)


def generate_consume_vectorized(consume_series: np.ndarray, n_simulations: int, cv: float = 0.05) -> np.ndarray:
    """向量化生成消费量矩阵"""
    n_rows = len(consume_series)
    result = np.zeros((n_rows, n_simulations))
    non_zero_mask = consume_series > 0

    if non_zero_mask.any():
        non_zero_values = consume_series[non_zero_mask][:, None]
        std = non_zero_values * cv
        a = (non_zero_values * 0.8 - non_zero_values) / std
        b = (non_zero_values * 1.2 - non_zero_values) / std

        generated_values = truncnorm.rvs(
            a, b, loc=non_zero_values, scale=std,
            size=(non_zero_mask.sum(), n_simulations)
        )
        result[non_zero_mask, :] = generated_values

    return result

def calculate_thq(
    df_pfas: pd.DataFrame,
    pfas: str,
    endfix: str,
    rfd_data: pd.Series,
    n_simulations: int = 100
) -> pd.DataFrame:
    """单个 PFAS 的 THQ计算"""
    n_rows = len(df_pfas)

    # TC
    tc_min = df_pfas[f'{pfas}_min'].values[:, None]
    tc_max = df_pfas[f'{pfas}_max'].values[:, None]
    tc_all = np.random.uniform(tc_min, tc_max, (n_rows, n_simulations))
    # 根据样品类型（endfix）选择单位转换
    if endfix == 'sw':  # 水样 (ng/L → mg/kg)
        tc_all = tc_all / 1_000_000
    else:               # 鱼样 (ng/g → mg/kg)
        tc_all = tc_all / 1000
    # RFD
    if rfd_data['min'] == rfd_data['max']:
        rfd_all = np.full(n_simulations, rfd_data['mode'] / 1e6)
    else:
        min_val = rfd_data['min'] / 1e6
        max_val = rfd_data['max'] / 1e6
        median_val = rfd_data['mode'] / 1e6
        c = (median_val - min_val) / (max_val - min_val)
        rfd_all = triang.rvs(c, loc=min_val, scale=max_val - min_val, size=n_simulations)

    # BW
    bw_mean = df_pfas['weight'].values[:, None]
    bw_std = bw_mean * 0.05
    a_bw = (bw_mean * 0.8 - bw_mean) / bw_std
    b_bw = (bw_mean * 1.2 - bw_mean) / bw_std
    bw_all = truncnorm.rvs(a_bw, b_bw, loc=bw_mean, scale=bw_std, size=(n_rows, n_simulations))

    # 寿命
    life_mean = df_pfas['life_ex'].values[:, None]
    life_std = life_mean * 0.05
    a_life = (life_mean * 0.8 - life_mean) / life_std
    b_life = (life_mean * 1.2 - life_mean) / life_std
    life_all = truncnorm.rvs(a_life, b_life, loc=life_mean, scale=life_std, size=(n_rows, n_simulations))

    ef = 365
    at_all = 365 * life_all

    # 消费量
    consume_base = df_pfas[f'{endfix}_consume'].values
    consume_all = generate_consume_vectorized(consume_base, n_simulations)

    thq_all = (tc_all * consume_all * ef * life_all) / (rfd_all * bw_all * at_all)

    # 一次性构造结果
    results = df_pfas[['lon', 'lat', 'year']].copy()
    thq_df = pd.DataFrame(
        thq_all,
        columns=[f'thq_{endfix}_{i}' for i in range(n_simulations)],
        index=results.index
    )
    results = pd.concat([results, thq_df], axis=1)

    return results



# =============================
# THQ计算模块（单个PFAS）
# =============================
def run_thq_for_one_pfas(
    pfas: str,
    source: str,
    mark: str,
    endfix: str,
    path_pfas_csv: str,
    path_result_nc: str,
    df_weight: pd.DataFrame,
    df_life: pd.DataFrame,
    df_rfd: pd.DataFrame,
    n_simulations: int = 100
):
    """运行单个 PFAS 的 THQ计算并保存为 NetCDF"""
    print(f"Processing {source} - {pfas}...")

    # safe_pfas_name = convert(pfas)
    df_pfas = pd.read_csv(os.path.join(path_pfas_csv, f'{endfix}_{pfas}_{mark}.csv'))
    df_pfas = df_pfas[df_pfas[f'{pfas}_min'].notna()]
    df_pfas = df_pfas.merge(df_weight, on='country_code', how='left')
    df_pfas = df_pfas.merge(df_life, on=['country_code', 'year'], how='left')

    if source == 'sw':
        df_pfas['sw_consume'] = 2

    rfd_data = df_rfd[df_rfd['PFAS'] == pfas].iloc[0]
    results = calculate_thq(df_pfas, pfas, endfix, rfd_data, n_simulations)
    if source == 'lr':
        thq_cols = [col for col in results.columns if col.startswith(f"thq_{endfix}_")]
        results[thq_cols] = results[thq_cols] * 0.71
    # 保存 NetCDF
    coords_lon = np.sort(results["lon"].unique())
    coords_lat = np.sort(results["lat"].unique())
    coords_year = np.sort(results["year"].unique())
    ds = xr.Dataset(coords={"year": coords_year, "lat": coords_lat, "lon": coords_lon})
    results_indexed = results.set_index(["year", "lat", "lon"])
    for col in results_indexed.columns:
        ds[col] = (("year", "lat", "lon"), results_indexed[col].to_xarray().values)
    for var in ds.data_vars:
        ds[var] = ds[var].round(3)
    os.makedirs(path_result_nc, exist_ok=True)
    output_file = os.path.join(path_result_nc, f'{endfix}_{pfas}_{mark}_thq.nc')
    ds.to_netcdf(output_file)

    print(f"✅ Saved: {output_file}")


# =============================
# 数据读取模块（nc -> CSV）
# =============================
def load_nc_and_save_csv(
    nc_path: str,
    df_grid: pd.DataFrame,
    nearest_lon: str,
    nearest_lat: str,
    list_remain: List[str],
    prefix: str,
    endfix: str,
    mark: str,
    pfas_name: str,
    path_part4_pfas: str
):
    """读取 nc 文件并保存为对应 PFAS 的 CSV"""
    ds = xr.open_dataset(nc_path)
    df_nc = ds[['lon', 'lat', 'year', 'max', 'min', 'mean']].to_dataframe().reset_index()
    df_nc.rename(columns={'lon': nearest_lon, 'lat': nearest_lat}, inplace=True)

    df_grid_pfas = df_grid.merge(df_nc,
                                 left_on=[nearest_lon, nearest_lat, 'year'],
                                 right_on=[nearest_lon, nearest_lat, 'year'],
                                 how='left')
    df_grid_pfas = df_grid_pfas[list_remain]
    df_grid_pfas.rename(columns={
        'min': f"{pfas_name}_min",
        'max': f"{pfas_name}_max",
        'mean': f"{pfas_name}_mean"
    }, inplace=True)

    csv_file = os.path.join(path_part4_pfas, f"{endfix}_{pfas_name}_{mark}.csv")
    os.makedirs(path_part4_pfas, exist_ok=True)
    df_grid_pfas.to_csv(csv_file, index=False)
    print(f"✅ CSV saved: {csv_file}")


# =============================
# 调度模块
# =============================
def process_nc_file(
    nc_dir: str,
    df_thq_param: pd.DataFrame,
    prefix: str,
    endfix: str,
    mark: str,
    nearest_lon: str,
    nearest_lat: str,
    list_remain: List[str],
    path_part4_pfas: str,
    list_pfas: List[str],
    start_year:int,
    end_year:int,
    run_thq_kwargs: dict,

):
    """处理整个 nc 文件目录，并运行 THQ计算"""
    start_year = start_year
    end_year = end_year
    years = list(range(start_year, end_year + 1))
    df_grid = df_thq_param[df_thq_param['year'].isin(years)].copy()
    nc_files = [f for f in os.listdir(nc_dir) if f.endswith('.nc')]
    for nc_file in nc_files:
        pfas_name = os.path.splitext(nc_file)[0].replace(f"{prefix}_", "")
        if pfas_name not in list_pfas:
            continue
        print(f"📂 Processing NC file: {prefix}_{pfas_name}")
        nc_path = os.path.join(nc_dir, nc_file)
        load_nc_and_save_csv(
            nc_path, df_grid, nearest_lon, nearest_lat, list_remain,
            prefix, endfix, mark, pfas_name, path_part4_pfas
        )
        # 运行单个PFAS的THQ
        run_thq_for_one_pfas(pfas=pfas_name, source=prefix, mark=mark, endfix=endfix, **run_thq_kwargs)



In [3]:

# 读取参数表，只需要一次
df_thq_param = pd.read_csv(path_part4_grid + 'thq_param.csv')
# 读取几组必要的数据，只需要一次
df_weight = pd.read_csv(path_part4_hi + 'weight.csv')  # ['country_code', 'weight']
df_life = pd.read_csv(path_part4_hi + 'life_ex.csv')  # ['country_code', 'year', 'life_ex']

df_rfd = pd.read_csv(path_part4_hi + 'rfd.csv')        # ['PFAS', 'mean', 'min', 'max']
print(df_rfd)
# 要保留的列
list_remain = ['lon','lat','year','country_code','ff_consume','sf_consume', 'min', 'max', 'mean']

list_pfas_all = df_rfd['PFAS'].tolist()
print(list_pfas_all)

list_pfas_lc = ['PFOA', 'PFNA', 'PFDA', 'PFUnDA','PFDoDA','PFTrDA', 'PFTeDA', 'PFHxS', 'PFOS', 'FOSA']
list_pfas_sc = ['PFBA', 'PFPeA', 'PFHxA', 'PFHpA','PFBS','HFPO-DA']

       PFAS        min     max     mode pfas_file_name
0      FOSA    12.0000    12.0    12.00           FOSA
1   HFPO-DA     3.0000    77.0    40.00        HFPO-DA
2      PFBA  2900.0000  3000.0  2950.00           PFBA
3      PFBS   300.0000  1400.0   430.00           PFBS
4      PFDA    12.0000    12.0    12.00           PFDA
5    PFDoDA    12.0000    12.0    12.00         PFDoDA
6     PFHpA    23.0000    23.0    23.00          PFHpA
7     PFHxA   500.0000   500.0   500.00          PFHxA
8     PFHxS     3.8000    20.0     6.85          PFHxS
9      PFNA     3.0000    12.0     4.30           PFNA
10     PFOA     0.0015    18.0     4.55           PFOA
11     PFOS     0.0079    23.0     2.50           PFOS
12    PFPeA   500.0000   500.0   500.00          PFPeA
13   PFTeDA    12.0000    12.0    12.00         PFTeDA
14   PFTrDA    12.0000    12.0    12.00         PFTrDA
15   PFUnDA    12.0000    12.0    12.00         PFUnDA
['FOSA', 'HFPO-DA', 'PFBA', 'PFBS', 'PFDA', 'PFDoDA', 'PFHpA', 'P

In [4]:
import pandas as pd
from joblib import Parallel, delayed
# 并行有亿点点吵
def run_one_row(p, df_thq_param, list_remain, pathf_part4_thq_csv,
                list_pfas_all, df_weight, df_life, df_rfd, pathf_part4_thq_rnc):
    """封装一行的处理逻辑"""
    return process_nc_file(
        nc_dir=p["nc_dir"],
        df_thq_param=df_thq_param,
        prefix=p["prefix"],
        endfix=p["endfix"],
        mark=p["mark"],
        nearest_lon=p["nearest_lon"],
        nearest_lat=p["nearest_lat"],
        list_remain=list_remain,
        path_part4_pfas=pathf_part4_thq_csv,
        list_pfas=list_pfas_all,
        start_year=int(p["start_year"]),
        end_year=int(p["end_year"]),
        run_thq_kwargs={
            "path_pfas_csv": pathf_part4_thq_csv,
            "path_result_nc": pathf_part4_thq_rnc,
            "df_weight": df_weight,
            "df_life": df_life,
            "df_rfd": df_rfd,
            "n_simulations": 100
        }
    )

if __name__ == "__main__":
    csv_path = r"F:\User_file\wyy\SPDB\part4_assess\start_thq.csv"
    # 这个csv文件指定路径
    df = pd.read_csv(csv_path)
    # n_jobs=-1 表示用尽可能多的 CPU 核心
    results = Parallel(n_jobs=6, backend="loky")(
        delayed(run_one_row)(
            p, df_thq_param, list_remain, pathf_part4_thq_csv,
            list_pfas_all, df_weight, df_life, df_rfd, pathf_part4_thq_rnc
        ) for _, p in df.iterrows()
    )
# 32 min

### merge_new

In [5]:
import os
import numpy as np
import xarray as xr
from joblib import Parallel, delayed


def process_one_group(input_dir, output_dir, prefix, endfix, group_name, pfas_list, mode="max"):
    """
    处理一个 (prefix, endfix, group_name) 的合并任务
    mode: "max" 表示取最大值, "sum" 表示求和
    """
    ds_result = None
    found_files = 0

    for pfas_name in pfas_list:
        file_name = f"{prefix}_{pfas_name}_{endfix}_thq.nc"
        file_path = os.path.join(input_dir, file_name)
        if not os.path.exists(file_path):
            print(f"[跳过] {file_path} 不存在")
            continue

        with xr.open_dataset(file_path) as ds:
            found_files += 1

            if ds_result is None:
                ds_result = ds.copy(deep=True)
            else:
                # 自动遍历所有变量
                for var_name in ds.data_vars:
                    if mode == "sum":
                        ds_result[var_name] = ds_result[var_name] + ds[var_name]
                    elif mode == "max":
                        ds_result[var_name] = xr.apply_ufunc(
                            np.maximum, ds_result[var_name], ds[var_name]
                        )
                    else:
                        raise ValueError(f"未知模式: {mode}")

    if ds_result is None:
        return f"[无数据] {prefix} {group_name} {endfix} 没找到文件"

    # 改名 prefix -> group_name
    rename_dict = {}
    for var_name in ds_result.data_vars:
        if var_name.startswith(f"thq_{prefix}_"):
            suffix = var_name.split(f"thq_{prefix}_", 1)[1]
            rename_dict[var_name] = f"thq_{group_name}_{suffix}"

    ds_result = ds_result.rename(rename_dict)
    for var_name in ds_result.data_vars:
        ds_result[var_name] = ds_result[var_name].round(2)
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"{prefix}_{group_name}_{endfix}_thq.nc")
    ds_result.to_netcdf(output_file)
    ds_result.close()

    return f"[完成] {output_file}, 合并了 {found_files} 个文件, 模式={mode}"


def merge_pfas_groups_parallel_joblib(
    input_dir,
    output_dir,
    prefixes,
    endfixes,
    groups_dict,
    n_jobs=4,
    mode="max"
):
    """用 joblib 并行执行合并任务"""
    tasks = (
        (prefix, endfix, group_name, pfas_list)
        for prefix in prefixes
        for endfix in endfixes
        for group_name, pfas_list in groups_dict.items()
    )

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(process_one_group)(input_dir, output_dir, prefix, endfix, group_name, pfas_list, mode)
        for prefix, endfix, group_name, pfas_list in tasks
    )

    for r in results:
        print(r)



In [6]:

if __name__ == "__main__":
    list_pfas_lc = ['PFOA', 'PFNA', 'PFDA', 'PFUnDA','PFDoDA','PFTrDA', 'PFTeDA', 'PFHxS', 'PFOS', 'FOSA']
    list_pfas_sc = ['PFBA', 'PFPeA', 'PFHxA', 'PFHpA','PFBS','HFPO-DA']
    list_pfas_all = list_pfas_lc + list_pfas_sc

    groups = {
        'lc': list_pfas_lc,
        'sc': list_pfas_sc,
        'all': list_pfas_all
    }
    # 'history', 'high', 'low', 'base', 'us']
    # mode 参数控制合并方式: "max" 取最大值, "sum" 求和
    merge_pfas_groups_parallel_joblib(
        input_dir=r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_raw",
        output_dir=r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge",
        prefixes=['sw', 'ff', 'sf'],
        endfixes=['history', 'high', 'low', 'base', 'us'],
        groups_dict=groups,
        n_jobs=6,
        mode="sum"  # 改成 "sum" 就是加和
    )

[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_lc_history_thq.nc, 合并了 10 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_sc_history_thq.nc, 合并了 6 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_all_history_thq.nc, 合并了 16 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_lc_high_thq.nc, 合并了 10 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_sc_high_thq.nc, 合并了 6 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_all_high_thq.nc, 合并了 16 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_lc_low_thq.nc, 合并了 10 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_sc_low_thq.nc, 合并了 6 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_all_low_thq.nc, 合并了 16 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_lc_base_thq.nc, 合并了 10 个文件, 模式=sum
[完成] F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge\sw_sc_base_thq.nc, 合并了 6 个文件, 模

### add

In [7]:
import xarray as xr
from pathlib import Path

def merge_nc_files(input_dir, output_dir, endfixe=['history', 'high', 'low', 'base']):
    """
    合并NC文件的主函数
    
    Parameters:
    -----------
    input_dir : str
        输入NC文件所在目录
    output_dir : str
        输出合并后NC文件的目录
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # 创建输出目录
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 定义参数
    groups = ['all', 'lc', 'sc']
    endfixes = endfixe  # ['history', 'high', 'low', 'base', 'us']
    
    # 第一步：合并 sf 和 ff 为 fish
    print("Step 1: 合并 sf 和 ff 为 fish...")
    merge_two_prefixes(input_path, output_path, 'sf', 'ff', 'fish', groups, endfixes)
    
    # 第二步：合并 fish 和 sw 为 total
    print("Step 2: 合并 fish 和 sw 为 total...")
    merge_two_prefixes(output_path, output_path, 'fish', 'sw', 'total', groups, endfixes, 
                      sw_from_input=True, input_path=input_path)
    
    print("合并完成！")


def merge_two_prefixes(path1, output_path, prefix1, prefix2, new_prefix, groups, endfixes, 
                       sw_from_input=False, input_path=None):
    """
    合并两个prefix的NC文件（使用矢量化运算）
    
    Parameters:
    -----------
    path1 : Path
        第一个prefix文件所在路径
    output_path : Path
        输出路径
    prefix1 : str
        第一个prefix名称
    prefix2 : str
        第二个prefix名称
    new_prefix : str
        合并后的新prefix名称
    groups : list
        group列表
    endfixes : list
        endfix列表
    sw_from_input : bool
        是否从原始输入目录读取sw文件（用于第二步合并）
    input_path : Path
        原始输入路径（当sw_from_input=True时使用）
    """
    for group in groups:
        for endfix in endfixes:
            # 构建文件名
            file1_name = f'{prefix1}_{group}_{endfix}_thq.nc'
            file2_name = f'{prefix2}_{group}_{endfix}_thq.nc'
            output_name = f'{new_prefix}_{group}_{endfix}_thq.nc'
            
            # 构建完整路径
            file1_path = path1 / file1_name
            
            # 如果是第二步合并，sw文件从原始输入目录读取
            if sw_from_input and prefix2 == 'sw':
                file2_path = input_path / file2_name
            else:
                file2_path = path1 / file2_name
            
            output_file_path = output_path / output_name
            
            # 检查文件是否存在
            if not file1_path.exists():
                print(f"警告: 文件不存在 - {file1_path}")
                continue
            if not file2_path.exists():
                print(f"警告: 文件不存在 - {file2_path}")
                continue
            
            try:
                # 读取NC文件
                with xr.open_dataset(file1_path) as ds1, xr.open_dataset(file2_path) as ds2:
                    # 使用矢量化运算：直接对整个数据集进行加法操作
                    # xarray会自动对齐坐标并对所有数据变量进行运算
                    ds_merged = ds1 + ds2
                    
                    # 保存合并后的文件
                    # 使用compute=True确保数据被计算后再保存
                    ds_merged.to_netcdf(output_file_path, compute=True)
                    
                print(f"成功合并: {output_name}")
                
            except Exception as e:
                print(f"错误: 合并 {file1_name} 和 {file2_name} 时出错 - {str(e)}")





In [8]:
# 使用示例
# ['history', 'high', 'low', 'base', 'us']
if __name__ == "__main__":
    input_directory = r'F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge'
    output_directory = r'F:\User_file\wyy\SPDB\part4_assess\thq\nc_add'
    
    merge_nc_files(input_directory, output_directory, ['history', 'high', 'low', 'base', 'us'])
# 3 min

Step 1: 合并 sf 和 ff 为 fish...
成功合并: fish_all_history_thq.nc
成功合并: fish_all_high_thq.nc
成功合并: fish_all_low_thq.nc
成功合并: fish_all_base_thq.nc
成功合并: fish_all_us_thq.nc
成功合并: fish_lc_history_thq.nc
成功合并: fish_lc_high_thq.nc
成功合并: fish_lc_low_thq.nc
成功合并: fish_lc_base_thq.nc
成功合并: fish_lc_us_thq.nc
成功合并: fish_sc_history_thq.nc
成功合并: fish_sc_high_thq.nc
成功合并: fish_sc_low_thq.nc
成功合并: fish_sc_base_thq.nc
成功合并: fish_sc_us_thq.nc
Step 2: 合并 fish 和 sw 为 total...
成功合并: total_all_history_thq.nc
成功合并: total_all_high_thq.nc
成功合并: total_all_low_thq.nc
成功合并: total_all_base_thq.nc
成功合并: total_all_us_thq.nc
成功合并: total_lc_history_thq.nc
成功合并: total_lc_high_thq.nc
成功合并: total_lc_low_thq.nc
成功合并: total_lc_base_thq.nc
成功合并: total_lc_us_thq.nc
成功合并: total_sc_history_thq.nc
成功合并: total_sc_high_thq.nc
成功合并: total_sc_low_thq.nc
成功合并: total_sc_base_thq.nc
成功合并: total_sc_us_thq.nc
合并完成！


### stat

In [9]:
from pathlib import Path
import xarray as xr
from joblib import Parallel, delayed

def summarize_nc(input_nc, output_nc):
    """
    读取一个nc文件，计算数据变量的统计量，并保存成新的nc文件
    """
    ds = xr.open_dataset(input_nc)
    data_vars = [v for v in ds.data_vars if v.startswith('thq_')]
    if not data_vars:
        print(f"文件 {input_nc} 中没有找到 thq_ 开头的数据变量，跳过。")
        ds.close()
        return

    da = xr.concat([ds[v] for v in data_vars], dim='member')

    stats = xr.Dataset()
    stats['thq_min'] = da.min(dim='member')
    stats['thq_max'] = da.max(dim='member')
    stats['thq_p2_5'] = da.quantile(0.025, dim='member')
    stats['thq_p50'] = da.quantile(0.5, dim='member')
    stats['thq_p97_5'] = da.quantile(0.975, dim='member')
    stats['thq_mean'] = da.mean(dim='member')
    stats['thq_std'] = da.std(dim='member')

    # 添加原始坐标
    for coord in ds.coords:
        stats = stats.assign_coords({coord: ds[coord]})

    stats.to_netcdf(output_nc)
    ds.close()
    print(f"保存统计文件到: {output_nc}")


In [10]:

if __name__ == "__main__":
    # 输入目录列表
    input_dirs = [
        r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_add",
        r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge"
    ]

    # 统一输出目录
    output_dir = Path(r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_stat")
    output_dir.mkdir(exist_ok=True)

    # 收集处理任务
    tasks = []
    for in_dir in input_dirs:
        in_path = Path(in_dir)
        for nc_file in in_path.glob("*.nc"):
            output_file = output_dir / nc_file.name  # 保留原文件名
            tasks.append((nc_file, output_file))

    # 并行执行
    Parallel(n_jobs=12)(delayed(summarize_nc)(input_nc, output_nc)
                        for input_nc, output_nc in tasks)

# 18 min

### pop

In [20]:
import os
import pandas as pd
import xarray as xr
from joblib import Parallel, delayed

def process_impact_population_nc(input_ds, param_ds, use_wiw=False):
    """处理影响人口（NetCDF版本）"""
    def get_suffix_after_second_underscore(var_name):
        parts = var_name.split('_', 2)
        return parts[2] if len(parts) >= 3 else var_name
    
    coord_vars = {'lon', 'lat', 'year'}
    thq_vars = [var for var in input_ds.data_vars if var not in coord_vars]
    
    pop_data = param_ds['pop_wiw'] if use_wiw else param_ds['pop']
    pop_data = pop_data.reindex_like(input_ds, method=None, fill_value=np.nan)
    
    result_vars = {}
    for var in thq_vars:
        suffix = get_suffix_after_second_underscore(var)
        new_var_name = f'pop_{suffix}'
        # 先处理 thq >= 1 和 thq < 1 的情况
        pop_values = xr.where(input_ds[var] >= 1, pop_data, 0)
        # 向上取整
        pop_values = np.ceil(pop_values)
        # 将 thq 为 NaN 的位置恢复为 NaN
        pop_values = xr.where(np.isnan(input_ds[var]), np.nan, pop_values)
        pop_values = pop_values.astype(np.float32)
        result_vars[new_var_name] = pop_values
    
    return xr.Dataset(result_vars, coords={k: input_ds[k] for k in ['lon', 'lat', 'year']})

def process_single_file(file_path, param_ds, output_dir, use_wiw=False):
    """处理单个nc文件并保存"""
    try:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_dir, file_name.replace('.nc', '_imp_pop.nc'))
        
        input_ds = xr.open_dataset(file_path)
        result_ds = process_impact_population_nc(input_ds, param_ds, use_wiw)
        result_ds.to_netcdf(output_path)
        
        input_ds.close()
        result_ds.close()
        return f"{file_name} 处理完成"
    except Exception as e:
        return f"{file_name} 处理失败: {e}"



In [12]:

if __name__ == "__main__":
    # 路径设置
    # path_part4_grid = r"F:\User_file\wyy\SPDB\part4_assess\grid"
    path_part4_pop = r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_merge"
    os.makedirs(path_part4_pop, exist_ok=True)
    
    path_merge = r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge"
    path_add = r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_add"
    
    # 读取 CSV 文件并转换为 xarray Dataset（只保留需要的列）
    csv_file = os.path.join(path_part4_grid, "thq_param.csv")
    df = pd.read_csv(csv_file, usecols=['lon', 'lat', 'year', 'pop', 'per_wiw'])
    param_ds = xr.Dataset.from_dataframe(df.set_index(['lon', 'lat', 'year']))
    
    # 计算 pop_wiw（这是个百分数，所以乘0.01）
    # param_ds['pop_wiw'] = param_ds['pop'] * param_ds['per_wiw'] * 0.01
    # 不使用per_wiw这个参数了
    param_ds['pop_wiw'] = param_ds['pop'] * 1
    # 收集两个路径下需要的文件
    nc_files_merge = [os.path.join(path_merge, f) for f in os.listdir(path_merge) if f.endswith(".nc")]
    nc_files_add_fish = [os.path.join(path_add, f) for f in os.listdir(path_add)
                         if f.endswith(".nc") and f.startswith("fish")]
    
    # 合并两个文件列表
    nc_files = nc_files_merge + nc_files_add_fish
    
    print(f"总共需要处理 {len(nc_files)} 个文件")
    print(f"  nc_merge 文件夹: {len(nc_files_merge)} 个")
    print(f"  nc_add   文件夹: {len(nc_files_add_fish)} 个（fish开头）")
    
    Parallel(n_jobs=6)(
        delayed(process_single_file)(
            file_path, 
            param_ds, 
            path_part4_pop, 
            use_wiw=os.path.basename(file_path).startswith("sw") # sw开头则启动
        )
        for file_path in nc_files
    )

总共需要处理 60 个文件
  nc_merge 文件夹: 45 个
  nc_add   文件夹: 15 个（fish开头）


In [23]:

if __name__ == "__main__":
    # 路径设置
    # path_part4_grid = r"F:\User_file\wyy\SPDB\part4_assess\grid"
    path_part4_pop = r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_merge"
    os.makedirs(path_part4_pop, exist_ok=True)
    
    path_add = r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_add"
    
    # 读取 CSV 文件并转换为 xarray Dataset（只保留需要的列）
    csv_file = os.path.join(path_part4_grid, "thq_param.csv")
    df = pd.read_csv(csv_file, usecols=['lon', 'lat', 'year', 'pop', 'per_wiw'])
    param_ds = xr.Dataset.from_dataframe(df.set_index(['lon', 'lat', 'year']))
    

    param_ds['pop_wiw'] = param_ds['pop'] * 1

    nc_files_add_fish = [os.path.join(path_add, f) for f in os.listdir(path_add)
                         if f.endswith(".nc") and f.startswith("total")]
    

    print(f"  nc_add   文件夹: {len(nc_files_add_fish)} 个（fish开头）")
    
    Parallel(n_jobs=6)(
        delayed(process_single_file)(
            file_path, 
            param_ds, 
            path_part4_pop, 
            use_wiw=os.path.basename(file_path).startswith("sw") # sw开头则启动
        )
        for file_path in nc_files_add_fish
    )

  nc_add   文件夹: 15 个（fish开头）


### add

这个好像可以退休了，不从pop这边算总和，直接从thq那边算

In [13]:
import xarray as xr
import numpy as np
import os

def merge_two_nc(file1, file2, group, endfix, dst_dir, n_simulations=100):
    """
    合并两个nc文件（file1 和 file2），取逐点最大值
    file1, file2 : nc文件路径
    group        : all / lc / sc
    endfix       : history / high / low / base / us
    dst_dir      : 输出目录
    """
    # 读取数据
    ds1 = xr.open_dataset(file1)
    ds2 = xr.open_dataset(file2)

    merged_data_vars = {}
    for i in range(n_simulations):
        var1 = ds1[f"pop_{i}"]
        var2 = ds2[f"pop_{i}"]
        # 用 np.maximum 代替 xr.ufuncs.maximum
        merged = np.maximum(var1, var2)  
        merged.name = f"pop_{group}_{i}"
        merged_data_vars[merged.name] = merged

    ds_out = xr.Dataset(
        data_vars=merged_data_vars,
        coords={c: ds1[c] for c in ("lon", "lat", "year")}
    )

    # 输出文件名
    out_name = f"total_{group}_{endfix}_thq_imp_pop.nc"
    os.makedirs(dst_dir, exist_ok=True)
    out_path = os.path.join(dst_dir, out_name)
    ds_out.to_netcdf(out_path)

    print(f"已生成: {out_path}")


In [14]:
from joblib import Parallel, delayed
import os

if __name__ == "__main__":
    src_dir = r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_merge"
    dst_dir = r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_add"

    files = os.listdir(src_dir)
    fish_files = [f for f in files if f.startswith('fish')]
    tasks = []
    for fish_file in fish_files:
        parts = fish_file.replace(".nc", "").split("_")
        _, group, endfix, *_ = parts
        sw_file = f"sw_{group}_{endfix}_thq_imp_pop.nc"

        fish_path = os.path.join(src_dir, fish_file)
        sw_path = os.path.join(src_dir, sw_file)

        if os.path.exists(sw_path):
            tasks.append((fish_path, sw_path, group, endfix))
        else:
            print(f"缺失对应文件: {sw_file}")

    # 并行执行
    Parallel(n_jobs=6)(
        delayed(merge_two_nc)(file1, file2, group, endfix, dst_dir, n_simulations=100)
        for file1, file2, group, endfix in tasks
    )


### stat

In [15]:
from pathlib import Path
import xarray as xr
from joblib import Parallel, delayed

def summarize_nc(input_nc, output_nc, var_name):
    """ 
    读取一个nc文件，计算数据变量的统计量，并保存成新的nc文件
    """
    ds = xr.open_dataset(input_nc)
    data_vars = [v for v in ds.data_vars if v.startswith('pop_')]
    if not data_vars:
        print(f"文件 {input_nc} 中没有找到 imp_ 开头的数据变量，跳过。")
        ds.close()
        return

    da = xr.concat([ds[v] for v in data_vars], dim='member')

    stats = xr.Dataset()
    stats[var_name + '_min'] = da.min(dim='member')
    stats[var_name + '_max'] = da.max(dim='member')
    stats[var_name + '_p2_5'] = da.quantile(0.025, dim='member')
    stats[var_name + '_p20'] = da.quantile(0.2, dim='member')
    stats[var_name + '_p40'] = da.quantile(0.4, dim='member')
    stats[var_name + '_p50'] = da.quantile(0.5, dim='member')
    stats[var_name + '_p60'] = da.quantile(0.6, dim='member')
    stats[var_name + '_p80'] = da.quantile(0.8, dim='member')
    stats[var_name + '_p97_5'] = da.quantile(0.975, dim='member')
    stats[var_name + '_mean'] = da.mean(dim='member')
    stats[var_name + '_std'] = da.std(dim='member')

    # 添加原始坐标
    for coord in ds.coords:
        stats = stats.assign_coords({coord: ds[coord]})

    stats.to_netcdf(output_nc)
    ds.close()
    print(f"保存统计文件到: {output_nc}")


In [24]:

if __name__ == "__main__":
    # 输入目录列表
    input_dirs = [
        r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_add",
        # r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_merge"
    ]

    # 统一输出目录
    output_dir = Path(r"F:\User_file\wyy\SPDB\part4_assess\pop\nc_stat")
    output_dir.mkdir(exist_ok=True)

    # 收集处理任务
    tasks = []
    for in_dir in input_dirs:
        in_path = Path(in_dir)
        for nc_file in in_path.glob("*.nc"):
            output_file = output_dir / nc_file.name  # 保留原文件名
            tasks.append((nc_file, output_file))

    # 并行执行
    Parallel(n_jobs=12)(delayed(summarize_nc)(input_nc, output_nc, 'pop')
                        for input_nc, output_nc in tasks)

# 18 min

### thq stat

In [19]:
import os
import numpy as np
import pandas as pd
import xarray as xr

# =========================
# 路径设置
# =========================
PATH_ADD = r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_add"
PATH_MERGE = r"F:\User_file\wyy\SPDB\part4_assess\thq\nc_merge"

OUT_RISK = r"F:\User_file\wyy\SPDB\part4_assess\thq\risk_prob"
OUT_PER_CONT = r"F:\User_file\wyy\SPDB\part4_assess\thq\per_cont"
OUT_SUMMARY = r"F:\User_file\wyy\SPDB\part4_assess\thq\summary_stats.csv"

os.makedirs(OUT_RISK, exist_ok=True)
os.makedirs(OUT_PER_CONT, exist_ok=True)

# =========================
# 参数设置
# =========================
SCENARIOS_RISK = ["base", "high", "history", "low"]
SCENARIOS_ALL = ["base", "high", "history", "low", "us"]

N_VARS = 100
THRESHOLD = 1.0


# =========================
# 工具函数
# =========================
def make_thq_vars(prefix, n=N_VARS):
    """
    根据变量前缀生成变量名列表
    如 prefix='all' -> thq_all_0 ~ thq_all_99
    """
    return [f"thq_{prefix}_{i}" for i in range(n)]


def get_prefix_from_filename(path):
    """
    根据 nc 文件名自动识别变量前缀。
    规则：取文件名按 "_" 分割后的第二个词条。
    
    示例：
    total_all_base_thq.nc -> all
    total_lc_base_thq.nc  -> lc
    total_sc_base_thq.nc  -> sc
    ff_all_base_thq.nc    -> all
    sf_all_base_thq.nc    -> all
    sw_all_base_thq.nc    -> all
    """
    name = os.path.basename(path)
    stem = os.path.splitext(name)[0]   # 去掉 .nc
    parts = stem.split("_")
    if len(parts) < 2:
        raise ValueError(f"无法从文件名识别变量前缀: {name}")
    return parts[1]


def get_var_names_from_file(path, n=N_VARS):
    """
    根据文件名自动生成该文件应有的变量名列表
    """
    prefix = get_prefix_from_filename(path)
    return make_thq_vars(prefix, n=n)


def round_da(da, n=2):
    return xr.apply_ufunc(lambda x: np.round(x, n), da)


def check_vars(ds, file_path, var_names):
    missing = [v for v in var_names if v not in ds.data_vars]
    if missing:
        raise ValueError(
            f"文件缺少变量: {file_path}\n"
            f"预期变量示例: {var_names[:10]}\n"
            f"缺少变量示例: {missing[:10]}\n"
            f"文件内实际变量示例: {list(ds.data_vars)[:10]}"
        )


def load_nc(path, var_names=None):
    """
    读取 nc，并检查变量是否完整。
    若未提供 var_names，则根据文件名自动识别。
    """
    if var_names is None:
        var_names = get_var_names_from_file(path)

    ds = xr.open_dataset(path)
    check_vars(ds, path, var_names)
    ds = ds.load()
    return ds


def save_dataset(ds, path):
    encoding = {}
    for v in ds.data_vars:
        encoding[v] = {"zlib": True, "complevel": 4}
    ds.to_netcdf(path, encoding=encoding)


def stack_runs(ds, var_names, run_dim="run"):
    """
    将指定变量列表堆叠成新的 run 维度
    输出维度类似: (run, lon, lat, year)
    """
    arr = xr.concat([ds[v] for v in var_names], dim=run_dim)
    arr = arr.assign_coords({run_dim: np.arange(len(var_names))})
    return arr


def stack_runs_from_file(ds, file_path, run_dim="run"):
    """
    根据文件名自动识别变量名前缀并堆叠
    """
    var_names = get_var_names_from_file(file_path)
    return stack_runs(ds, var_names, run_dim=run_dim)


def get_lat_weights(ds):
    """
    返回纬度权重 cos(lat)
    """
    if "lat" not in ds.coords:
        raise ValueError("数据中缺少 lat 坐标，无法进行面积加权")
    weights = np.cos(np.deg2rad(ds["lat"]))
    weights.name = "weights"
    return weights


def weighted_global_mean_by_year(da, ds_ref):
    """
    对 DataArray(lon, lat, year) 做基于纬度权重的全球平均
    返回 DataArray(year)
    """
    weights = get_lat_weights(ds_ref)
    out = da.weighted(weights).mean(dim=("lon", "lat"), skipna=True)
    return out


def yearly_summary_df(ds, var_name, scenario):
    """
    计算某变量每年的面积加权全球平均
    返回 DataFrame(year, scenario, metric, value)
    """
    mean_da = weighted_global_mean_by_year(ds[var_name], ds)
    df = mean_da.to_dataframe(name="value").reset_index()
    df["scenario"] = scenario
    df["metric"] = var_name
    df["value"] = df["value"].round(2)
    return df[["year", "scenario", "metric", "value"]]


# =========================
# 需求1：风险概率
# =========================
def task1_risk_probability():
    print("开始执行需求1：计算风险概率")
    summary_rows = []

    for scenario in SCENARIOS_RISK:
        in_file = os.path.join(PATH_ADD, f"total_all_{scenario}_thq.nc")
        out_file = os.path.join(OUT_RISK, f"risk_prob_total_all_{scenario}_thq.nc")

        if not os.path.exists(in_file):
            print(f"[跳过] 文件不存在: {in_file}")
            continue

        print(f"[处理中] {os.path.basename(in_file)}")
        ds = load_nc(in_file)

        arr_all = stack_runs_from_file(ds, in_file)   # 自动识别 thq_all_*
        risk_prob = (arr_all >= THRESHOLD).mean(dim="run", skipna=True)
        risk_prob = round_da(risk_prob, 2)

        out_ds = xr.Dataset(
            {"risk_prob": risk_prob},
            coords={k: ds.coords[k] for k in ["lon", "lat", "year"] if k in ds.coords}
        )

        save_dataset(out_ds, out_file)
        print(f"[完成] 风险概率已保存: {out_file}")

        summary_rows.append(yearly_summary_df(out_ds, "risk_prob", scenario))

        ds.close()

    if summary_rows:
        return pd.concat(summary_rows, ignore_index=True)
    else:
        return pd.DataFrame(columns=["year", "scenario", "metric", "value"])


# =========================
# 需求2：长短链贡献
# =========================
def task2_chain_contribution():
    print("开始执行需求2：计算长短链贡献")

    all_csv_rows = []
    summary_rows = []

    for scenario in SCENARIOS_ALL:
        file_all = os.path.join(PATH_ADD, f"total_all_{scenario}_thq.nc")
        file_lc = os.path.join(PATH_ADD, f"total_lc_{scenario}_thq.nc")
        file_sc = os.path.join(PATH_ADD, f"total_sc_{scenario}_thq.nc")

        if not (os.path.exists(file_all) and os.path.exists(file_lc) and os.path.exists(file_sc)):
            print(f"[跳过] 场景文件不完整: {scenario}")
            continue

        print(f"[处理中] 长短链贡献: {scenario}")

        ds_all = load_nc(file_all)
        ds_lc = load_nc(file_lc)
        ds_sc = load_nc(file_sc)

        arr_all = stack_runs_from_file(ds_all, file_all)  # thq_all_*
        arr_lc = stack_runs_from_file(ds_lc, file_lc)     # thq_lc_*
        arr_sc = stack_runs_from_file(ds_sc, file_sc)     # thq_sc_*

        # 逐次模拟筛选后再算比例
        lc_ratio = xr.where((arr_all >= THRESHOLD) & (arr_all != 0), arr_lc / arr_all, np.nan)
        sc_ratio = xr.where((arr_all >= THRESHOLD) & (arr_all != 0), arr_sc / arr_all, np.nan)

        lc_contrib = lc_ratio.mean(dim="run", skipna=True)
        sc_contrib = sc_ratio.mean(dim="run", skipna=True)

        lc_contrib = round_da(lc_contrib, 2)
        sc_contrib = round_da(sc_contrib, 2)

        out_ds = xr.Dataset(
            {
                "lc_contrib": lc_contrib,
                "sc_contrib": sc_contrib
            },
            coords={k: ds_all.coords[k] for k in ["lon", "lat", "year"] if k in ds_all.coords}
        )

        out_nc = os.path.join(OUT_PER_CONT, f"chain_contrib_{scenario}.nc")
        save_dataset(out_ds, out_nc)
        print(f"[完成] 长短链贡献nc已保存: {out_nc}")

        df = out_ds.to_dataframe().reset_index()
        df["scenario"] = scenario
        out_csv = os.path.join(OUT_PER_CONT, f"chain_contrib_{scenario}.csv")
        df.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[完成] 长短链贡献csv已保存: {out_csv}")

        all_csv_rows.append(df)

        summary_rows.append(yearly_summary_df(out_ds, "lc_contrib", scenario))
        summary_rows.append(yearly_summary_df(out_ds, "sc_contrib", scenario))

        ds_all.close()
        ds_lc.close()
        ds_sc.close()

    if all_csv_rows:
        all_df = pd.concat(all_csv_rows, ignore_index=True)
        all_df.to_csv(
            os.path.join(OUT_PER_CONT, "chain_contrib_all_scenarios.csv"),
            index=False,
            encoding="utf-8-sig"
        )

    if summary_rows:
        return pd.concat(summary_rows, ignore_index=True)
    else:
        return pd.DataFrame(columns=["year", "scenario", "metric", "value"])


# =========================
# 需求3：不同来源贡献
# =========================
def task3_source_contribution():
    print("开始执行需求3：计算不同来源贡献")

    all_csv_rows = []
    summary_rows = []

    for scenario in SCENARIOS_ALL:
        file_total = os.path.join(PATH_ADD, f"total_all_{scenario}_thq.nc")
        file_ff = os.path.join(PATH_MERGE, f"ff_all_{scenario}_thq.nc")
        file_sf = os.path.join(PATH_MERGE, f"sf_all_{scenario}_thq.nc")
        file_sw = os.path.join(PATH_MERGE, f"sw_all_{scenario}_thq.nc")

        if not (os.path.exists(file_total) and os.path.exists(file_ff) and os.path.exists(file_sf) and os.path.exists(file_sw)):
            print(f"[跳过] 场景文件不完整: {scenario}")
            continue

        print(f"[处理中] 来源贡献: {scenario}")

        ds_total = load_nc(file_total)
        ds_ff = load_nc(file_ff)
        ds_sf = load_nc(file_sf)
        ds_sw = load_nc(file_sw)

        arr_total = stack_runs_from_file(ds_total, file_total)  # thq_all_*
        arr_ff = stack_runs_from_file(ds_ff, file_ff)           # thq_all_*
        arr_sf = stack_runs_from_file(ds_sf, file_sf)           # thq_all_*
        arr_sw = stack_runs_from_file(ds_sw, file_sw)           # thq_all_*

        # 逐次模拟筛选后再算比例
        ff_ratio = xr.where((arr_total >= THRESHOLD) & (arr_total != 0), arr_ff / arr_total, np.nan)
        sf_ratio = xr.where((arr_total >= THRESHOLD) & (arr_total != 0), arr_sf / arr_total, np.nan)
        sw_ratio = xr.where((arr_total >= THRESHOLD) & (arr_total != 0), arr_sw / arr_total, np.nan)

        ff_contrib = ff_ratio.mean(dim="run", skipna=True)
        sf_contrib = sf_ratio.mean(dim="run", skipna=True)
        sw_contrib = sw_ratio.mean(dim="run", skipna=True)

        ff_contrib = round_da(ff_contrib, 2)
        sf_contrib = round_da(sf_contrib, 2)
        sw_contrib = round_da(sw_contrib, 2)

        out_ds = xr.Dataset(
            {
                "ff_contrib": ff_contrib,
                "sf_contrib": sf_contrib,
                "sw_contrib": sw_contrib
            },
            coords={k: ds_total.coords[k] for k in ["lon", "lat", "year"] if k in ds_total.coords}
        )

        out_nc = os.path.join(OUT_PER_CONT, f"source_contrib_{scenario}.nc")
        save_dataset(out_ds, out_nc)
        print(f"[完成] 不同来源贡献nc已保存: {out_nc}")

        df = out_ds.to_dataframe().reset_index()
        df["scenario"] = scenario
        out_csv = os.path.join(OUT_PER_CONT, f"source_contrib_{scenario}.csv")
        df.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[完成] 不同来源贡献csv已保存: {out_csv}")

        all_csv_rows.append(df)

        summary_rows.append(yearly_summary_df(out_ds, "ff_contrib", scenario))
        summary_rows.append(yearly_summary_df(out_ds, "sf_contrib", scenario))
        summary_rows.append(yearly_summary_df(out_ds, "sw_contrib", scenario))

        ds_total.close()
        ds_ff.close()
        ds_sf.close()
        ds_sw.close()

    if all_csv_rows:
        all_df = pd.concat(all_csv_rows, ignore_index=True)
        all_df.to_csv(
            os.path.join(OUT_PER_CONT, "source_contrib_all_scenarios.csv"),
            index=False,
            encoding="utf-8-sig"
        )

    if summary_rows:
        return pd.concat(summary_rows, ignore_index=True)
    else:
        return pd.DataFrame(columns=["year", "scenario", "metric", "value"])


# =========================
# 需求4：汇总输出
# =========================
def task4_summary(risk_summary, chain_summary, source_summary):
    print("开始执行需求4：统计汇总")

    frames = []
    if not risk_summary.empty:
        frames.append(risk_summary)
    if not chain_summary.empty:
        frames.append(chain_summary)
    if not source_summary.empty:
        frames.append(source_summary)

    if not frames:
        print("[提示] 没有可汇总的数据")
        return

    summary_df = pd.concat(frames, ignore_index=True)
    summary_df["value"] = summary_df["value"].round(2)
    summary_df = summary_df.sort_values(by=["scenario", "year", "metric"]).reset_index(drop=True)
    summary_df.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")
    print(f"[完成] 汇总统计已保存: {OUT_SUMMARY}")


# =========================
# 主程序
# =========================
if __name__ == "__main__":
    print("全部任务开始运行")

    risk_summary = task1_risk_probability()
    chain_summary = task2_chain_contribution()
    source_summary = task3_source_contribution()

    task4_summary(risk_summary, chain_summary, source_summary)

    print("全部任务执行完成")

全部任务开始运行
开始执行需求1：计算风险概率
[处理中] total_all_base_thq.nc
[完成] 风险概率已保存: F:\User_file\wyy\SPDB\part4_assess\thq\risk_prob\risk_prob_total_all_base_thq.nc
[处理中] total_all_high_thq.nc
[完成] 风险概率已保存: F:\User_file\wyy\SPDB\part4_assess\thq\risk_prob\risk_prob_total_all_high_thq.nc
[处理中] total_all_history_thq.nc
[完成] 风险概率已保存: F:\User_file\wyy\SPDB\part4_assess\thq\risk_prob\risk_prob_total_all_history_thq.nc
[处理中] total_all_low_thq.nc
[完成] 风险概率已保存: F:\User_file\wyy\SPDB\part4_assess\thq\risk_prob\risk_prob_total_all_low_thq.nc
开始执行需求2：计算长短链贡献
[处理中] 长短链贡献: base
[完成] 长短链贡献nc已保存: F:\User_file\wyy\SPDB\part4_assess\thq\per_cont\chain_contrib_base.nc
[完成] 长短链贡献csv已保存: F:\User_file\wyy\SPDB\part4_assess\thq\per_cont\chain_contrib_base.csv
[处理中] 长短链贡献: high
[完成] 长短链贡献nc已保存: F:\User_file\wyy\SPDB\part4_assess\thq\per_cont\chain_contrib_high.nc
[完成] 长短链贡献csv已保存: F:\User_file\wyy\SPDB\part4_assess\thq\per_cont\chain_contrib_high.csv
[处理中] 长短链贡献: history
[完成] 长短链贡献nc已保存: F:\User_file\wyy\SPDB\part4_assess\thq